# Ceftriaxone — Species-Masked Federated Learning

**Federated species masking per site, then compare 3 mask strategies + unmasked.**

| Phase | Content |
|---|---|
| 1 | Per-site RF trained to classify species → saved for 08 reuse |
| 2 | 3 mask strategies (union, majority, persite) + unmasked baseline |
| 3 | Mask comparison: centralized MLP + FedAvg MLP for each mask |
| 4 | Best mask → full FL pipeline (FedProx μ=0.5, 1.0, FedLR, FedRF) |
| Output | RF models + masks saved for 08 pipeline reuse |

All species (no E. coli filter). Species-stratified splits.

Reference implementation for the documentation page
[docs/06b](https://asd-standard.github.io/Dryad_DRIAMS/06b-Ceftriaxone-E-coli/index.html).

## Notes
- **Species masking vs the main federated run** — 06-03a keeps only *E. coli*
  spectra; this notebook keeps all species and instead zeroes each site's
  top-500 species-predictive m/z bins (``union``: any site, ``majority``:
  at least 2 sites, ``persite``: each site's own top-500) plus the unmasked
  ``none`` baseline. The winning mask -- highest worst-site FedAvg
  improvement over unmasked -- is then used by the full FL pipeline and the
  cross-site baselines.
- **Protocol shape** — one Flower client per DRIAMS site (A/B/C/D) over
  50 MLP rounds (5 for the tree-collection FedRF); per-site species-stratified
  90/10 split, per-site log1p+standardize, and training downsampled to keep
  all resistant spectra (susceptibles capped at 10x R in bad-ratio species).
- **Outputs** — under
  ``.../Analysis/06b-Ceftriaxone-E-coli/06-03c-Ceftriaxone-Federated-z-Runs/01-Run``
  (the path ``06-03d`` reads its species RFs and masks from):
  ``results/`` (per-round CSVs, ``final_results.csv``,
  ``best_params_used.txt``, convergence/loss/mask-delta/heatmap PDFs),
  ``models/`` (FedProx round checkpoints), ``masks/``
  (union/majority/persite arrays) and ``rf_species_models/`` (per-site
  species RFs kept for 08 reuse).
- **Provenance** — the federated numbers on the docs page come from the
  archived pre-fix run (30 rounds, mu=0.1, unsquared proximal penalty); this
  notebook now runs the corrected protocol (50 rounds, mu in {0.5, 1.0},
  squared proximal term). The tables will be refreshed after the next run.

In [ ]:
# ── CONFIG ──
# Run-level knobs: mask size (MASK_TOP_K species-predictive bins per site),
# federated rounds for the MLP strategies (NUM_ROUNDS) and for the
# tree-collection RF (NUM_RF_ROUNDS), plus the output folder name and the
# sub-run (TARGET_RUN) inside it.
MASK_TOP_K = 500
NUM_ROUNDS = 50
NUM_RF_ROUNDS = 5
RUN_NAME = "06-03c-Ceftriaxone-Federated-z-Runs"
TARGET_RUN = "01-Run"  # subdirectory within RUN_NAME

In [ ]:
# Colab dependency install for Flower (flwr[simulation]), MaldiDeepKit,
# MaldiMrKit and seaborn; a no-op when the stack is already present locally.
!pip install "flwr[simulation]" maldideepkit maldiamrkit seaborn --quiet

In [ ]:
# Mounts Google Drive on Colab so DRYAD can point at the Drive copy of the
# dataset; locally the import fails and IN_COLAB stays False.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# Imports (numpy/pandas/torch/sklearn/Flower/MaldiDeepKit) plus the warning
# filters and the global seed (42) applied to numpy and torch; Flower clients
# stay on CPU (FL_DEVICE) while torch uses CUDA when available (DEV).
import warnings, os, io, shutil, json as _json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     cross_val_predict, StratifiedKFold)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score, log_loss)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

import flwr as fl
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
FL_DEVICE = "cpu"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Flower: {fl.__version__}  |  Device: {DEV}")

In [ ]:
# DRYAD root (Colab vs local), drug name and the run directory layout:
# results/, models/, masks/ and rf_species_models/ under TARGET_RUN, plus the
# four per-site data.csv paths. SITE_ORDER fixes the client id -> site mapping
# used by every federated simulation below.
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"

PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06b-Ceftriaxone-E-coli" / RUN_NAME
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"; MODEL_DIR = RUN_DIR / "models"
MASK_DIR = RUN_DIR / "masks"; RF_DIR = RUN_DIR / "rf_species_models"
for d in [OUT_DIR, MODEL_DIR, MASK_DIR, RF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Drug: {DRUG_NAME}  |  Run: {RUN_DIR}")

In [ ]:
# ── Shared constants ──
# Tuning grids and FL knobs shared by all phases: threshold sweep, MLP
# lr/dropout grids, FedProx mu values, mask strategies, RF grid, and the
# per-species downsampling caps (keep all R, cap S at TRAIN_DS_S_PER_R x R
# in bad-ratio species) together with the helpers n_mixes/weighted_avg_loss.
THRESHOLDS = np.linspace(0.05, 0.95, 91)
LR_GRID = np.logspace(np.log10(3e-5), np.log10(2e-4), 8)
DROP_GRID = np.linspace(0.2, 0.8, 7)
LOCAL_EPOCHS = 1; BATCH_SIZE = 16
FEDPROX_MUS = [0.5, 1.0]
MASK_STRATEGIES = ["none", "union", "majority", "persite"]

RF_PARAM_GRID = {
    "n_estimators": [200, 300],
    "max_depth": [20, None],
    "min_samples_leaf": [5, 10],
    "class_weight": ["balanced", "balanced_subsample"],
}
C_GRID = np.linspace(5e-5, 1e-3, 15)

# ── Downsampling (keep all resistant; cap susceptibles per species) ──
TRAIN_DS_S_PER_R = 10   # global: all training data (bad-ratio species only)
RF_DS_MAX_SAMPLES = 3000  # RF grid-search-only: cap tuning samples (stratified by label)
RF_DS_BAD_RATIO = 0.10
RF_DS_MIN_N = 400

def n_mixes(n_train):
    """Local passes for a client of ``n_train`` samples: 5/4/3/1 by size."""
    if n_train < 500: return 5
    if n_train < 1500: return 4
    if n_train < 5000: return 3
    return 1

def weighted_avg_loss(metrics_list):
    """Sample-weighted mean of client ``train_loss`` metrics for Flower.

    Parameters
    ----------
    metrics_list : list
        Either ``(num_examples, metrics_dict)`` tuples or plain metrics dicts
        (``num_examples`` defaults to 1). Each dict should carry ``train_loss``.

    Returns
    -------
    dict
        ``{"avg_train_loss": float, "total_examples": int}``, or ``{}`` when
        the total weight is zero.
    """
    total = 0; weighted_sum = 0
    for item in metrics_list:
        n, d = item if isinstance(item, tuple) else (item.get("num_examples", 1), item)
        total += n; weighted_sum += d.get("train_loss", 0) * n
    return {"avg_train_loss": weighted_sum / total, "total_examples": total} if total > 0 else {}

In [ ]:
# ── Load Ceftriaxone — ALL species ──
# Reads each site's data.csv and keeps every spectrum regardless of species
# (the species column is kept for the stratified split and the Phase-1 RF);
# bin_* columns become the 6000-bin feature matrix. Prints per-site class
# balance and species counts.
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    nr, ns = (y==1).sum(), (y==0).sum()
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples ({ns} S, {nr} R, {nr/len(y)*100:.1f}% R, {n_sp} species)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")

In [ ]:
# ── Per-site species-stratified 90/10 split ──
# Each site is split exactly once (seed 42, species-stratified) into the
# local train/test pair a federated client would see; the test half stays
# untouched until evaluation. Pooled train arrays are stacked for the
# centralized baselines.
client_train = {}; client_test = {}
species_train = {}; species_test = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    n = len(y); idx = np.arange(n).reshape(-1,1)
    itr, iv, _, _ = stratified_species_drug_split(idx, y, species=sp, test_size=0.10, random_state=SEED)
    itr = itr.flatten().astype(int); iv = iv.flatten().astype(int)
    client_train[site] = (X[itr], y[itr]); client_test[site] = (X[iv], y[iv])
    species_train[site] = sp[itr]; species_test[site] = sp[iv]
    print(f"  Site {site}: train={len(itr)} test={len(iv)} species={len(np.unique(sp[itr]))}")

pooled_X_train = np.concatenate([client_train[s][0] for s in SITE_ORDER])
pooled_y_train = np.concatenate([client_train[s][1] for s in SITE_ORDER])
pooled_sp_train = np.concatenate([species_train[s] for s in SITE_ORDER])
print(f"\nPooled train: {len(pooled_X_train)} samples, {len(np.unique(pooled_sp_train))} species")

In [ ]:
# ── Downsample training data (keep all R, cap S in bad-ratio species) ──
# Class-imbalance control applied to the training halves only: every
# resistant spectrum is kept, susceptibles are capped at TRAIN_DS_S_PER_R x R
# for species whose resistant fraction is below RF_DS_BAD_RATIO. Test splits
# keep the full spectrum distribution.
def downsample_keep_idx(y, sp, s_per_r, bad_ratio=0.10, min_n=400, rng=None):
    """Keep all resistant spectra; cap susceptibles in bad-ratio species.

    For every species with a resistant fraction below ``bad_ratio`` (and more
    than ``min_n`` samples), susceptibles are subsampled to ``s_per_r`` per
    resistant so the local training set does not drown in negative examples.
    Species that do not qualify are kept in full.

    Parameters
    ----------
    y : ndarray (N,)
        Binary resistance labels (1 = resistant).
    sp : ndarray (N,) str
        Species name per spectrum.
    s_per_r : int
        Maximum susceptible-to-resistant ratio kept per species.
    bad_ratio : float, default=0.10
        Resistant fraction below which a species is considered badly balanced.
    min_n : int, default=400
        Species smaller than this are never downsampled.
    rng : numpy.random.Generator or None
        Seeded RNG for the subsampling (defaults to the notebook seed).

    Returns
    -------
    ndarray of int
        Sorted indices of the kept training rows.
    """
    rng = rng if rng is not None else np.random.default_rng(SEED)
    idx = np.arange(len(y))
    keep = idx[y == 1].tolist()
    for spec in np.unique(sp):
        m = sp == spec
        n_r = int((m & (y == 1)).sum()); n_s = int((m & (y == 0)).sum())
        total = n_r + n_s
        s_pos = idx[m & (y == 0)]
        if n_r > 0 and total > min_n and (n_r / total) < bad_ratio and n_s > n_r * s_per_r:
            keep.extend(rng.choice(s_pos, size=int(round(n_r * s_per_r)), replace=False).tolist())
        else:
            keep.extend(s_pos.tolist())
    return np.sort(np.array(keep, dtype=int))

_rng_ds = np.random.default_rng(SEED)
for site in SITE_ORDER:
    X, y = client_train[site]; sp = species_train[site]
    keep = downsample_keep_idx(y, sp, TRAIN_DS_S_PER_R, rng=_rng_ds)
    client_train[site] = (X[keep], y[keep]); species_train[site] = sp[keep]
    print(f"  Site {site}: kept {len(keep)}/{len(y)} train (R={(y[keep]==1).sum()}, S={(y[keep]==0).sum()})")

pooled_X_train = np.concatenate([client_train[s][0] for s in SITE_ORDER])
pooled_y_train = np.concatenate([client_train[s][1] for s in SITE_ORDER])
pooled_sp_train = np.concatenate([species_train[s] for s in SITE_ORDER])
print(f"Downsampled pooled train: {len(pooled_X_train)} samples")

In [ ]:
# ── Per-site preprocessing ──
# log1p + standardize fitted on each site's own training split (the Flower
# client's private transform) and applied to that site's test split. The data
# is still unmasked here; masks are applied in Phase 2.
client_train_pp = {}; client_test_pp = {}

for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
print("Per-site preprocessing done. Data still unmasked at this point.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 1: Per-site RF species models + save
# ═══════════════════════════════════════════════════════════════════════════
# Trains one RandomForest per site to predict species from the spectra; its
# top-MASK_TOP_K Gini-important bins are that site's candidate species mask.
# Sites with fewer than 3 species get a uniform importance vector and no mask.
# The fitted RFs are saved under rf_species_models/ for 08 reuse.

print("\n=== Phase 1: Per-Site RF Species Models ===")
site_importances = {}
site_masks_raw = {}   # top-K indices per site
site_n_species = {}

for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    sp_tr = species_train[site]
    n_sp = len(np.unique(sp_tr)); site_n_species[site] = n_sp

    if n_sp < 3:
        print(f"  Site {site}: only {n_sp} species — species masking would be meaningless")
        site_importances[site] = np.ones(6000) / 6000  # uniform
        site_masks_raw[site] = np.array([], dtype=int)
        continue

    rf_sp = RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_leaf=5,
                                    n_jobs=-1, oob_score=True, random_state=SEED)
    rf_sp.fit(X_tr, sp_tr)
    # Save for 08 reuse
    joblib.dump(rf_sp, RF_DIR / f"site_{site}_rf_species.joblib")

    site_importances[site] = rf_sp.feature_importances_
    top_k = np.argsort(rf_sp.feature_importances_)[-MASK_TOP_K:]
    top_k.sort()
    site_masks_raw[site] = top_k
    explained = rf_sp.feature_importances_[top_k].sum()
    print(f"  Site {site}: {n_sp} species, OOB acc={rf_sp.oob_score_:.4f}, "
          f"top-{MASK_TOP_K} bins explain {explained:.1%} of species importance")

print("\nPhase 1 done. RF models saved to rf_species_models/.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 2: Compute mask strategies + create masked data copies
# ═══════════════════════════════════════════════════════════════════════════
# Turns the per-site top-500 masks into three strategies: union (any site),
# majority (flagged by at least 2 sites) and persite (each site keeps its own).
# All masks are saved as .npy under masks/ and the pairwise site overlaps are
# printed as a quick sanity check.

print("\n=== Phase 2: Mask Computation ===")

# ── Union mask: any bin flagged by any site ──
all_site_bins = [site_masks_raw[s] for s in SITE_ORDER if len(site_masks_raw[s]) > 0]
union_mask = np.unique(np.concatenate(all_site_bins)) if all_site_bins else np.array([], dtype=int)
print(f"  Union mask: {len(union_mask)} bins")

# ── Majority mask: bin flagged by >= 2 sites ──
bin_counts = {}
for bins in all_site_bins:
    for b in bins:
        bin_counts[b] = bin_counts.get(b, 0) + 1
majority_mask = np.array([b for b, c in bin_counts.items() if c >= 2], dtype=int)
majority_mask.sort()
print(f"  Majority mask: {len(majority_mask)} bins (flagged by >=2 sites)")

# ── Persite mask: each site uses its own top-K (already stored in site_masks_raw) ──
print(f"  Persite mask: {MASK_TOP_K} bins per site")

# Save all masks
np.save(MASK_DIR / "union_mask.npy", union_mask)
np.save(MASK_DIR / "majority_mask.npy", majority_mask)
for site in SITE_ORDER:
    np.save(MASK_DIR / f"persite_site_{site}_mask.npy", site_masks_raw[site])

# ── Overlap between sites ──
print("\n  Pairwise mask overlaps (% shared bins):")
sites_with_mask = [s for s in SITE_ORDER if len(site_masks_raw[s]) > 0]
for i, sa in enumerate(sites_with_mask):
    for j, sb in enumerate(sites_with_mask):
        if j <= i: continue
        overlap = len(set(site_masks_raw[sa]) & set(site_masks_raw[sb]))
        pct = overlap / MASK_TOP_K * 100
        print(f"    {sa}∩{sb}: {overlap}/{MASK_TOP_K} ({pct:.0f}%)")

In [ ]:
# ── Create masked data variants for each strategy ──
# Copies the preprocessed (still unmasked) train/test/pooled arrays and
# zeroes the masked bins per strategy, so every subsequent model sees the
# same split with only those columns removed. "none" is a straight copy and
# stays the reference for the mask deltas computed in Phase 3.
# We keep the unmasked data in client_train_pp_unmasked for comparison

client_train_unmasked = {s: (client_train_pp[s][0].copy(), client_train_pp[s][1].copy())
                          for s in SITE_ORDER}
client_test_unmasked  = {s: (client_test_pp[s][0].copy(), client_test_pp[s][1].copy())
                          for s in SITE_ORDER}

# pooled preprocess for centralized
state_pool = fit_input_transform(pooled_X_train, "log1p+standardize")
X_pool_unmasked = apply_input_transform(pooled_X_train, state_pool).copy()

# Build test sets from preprocessed client_test
test_dict_unmasked = {s: (client_test_unmasked[s][0], client_test_unmasked[s][1]) for s in SITE_ORDER}
combined_X_test_unmasked = np.concatenate([test_dict_unmasked[s][0] for s in SITE_ORDER])
combined_y_test = np.concatenate([test_dict_unmasked[s][1] for s in SITE_ORDER])

# Masked variants
masked_data = {}  # {strategy: {train: {site: (X,y)}, test: {site: (X,y)}, pooled_X, pooled_y}}

for strategy in MASK_STRATEGIES:
    if strategy == "none":
        # Use unmasked data directly
        masked_data[strategy] = {"train": client_train_unmasked, "test": client_test_unmasked,
                                  "pooled_X": X_pool_unmasked, "pooled_y": pooled_y_train}
        continue

    # Determine the mask for this strategy
    if strategy == "union":
        global_mask = union_mask
    elif strategy == "majority":
        global_mask = majority_mask

    tr = {}; te = {}
    for site in SITE_ORDER:
        if strategy == "persite":
            mask = site_masks_raw[site] if len(site_masks_raw[site]) > 0 else np.array([], dtype=int)
        else:
            mask = global_mask
        if len(mask) == 0:
            tr[site] = (client_train_unmasked[site][0].copy(), client_train_unmasked[site][1].copy())
            te[site] = (client_test_unmasked[site][0].copy(), client_test_unmasked[site][1].copy())
        else:
            X_tr = client_train_unmasked[site][0].copy(); X_tr[:, mask] = 0.0
            X_te = client_test_unmasked[site][0].copy(); X_te[:, mask] = 0.0
            tr[site] = (X_tr, client_train_unmasked[site][1].copy())
            te[site] = (X_te, client_test_unmasked[site][1].copy())

    X_pool = X_pool_unmasked.copy()
    if len(global_mask if strategy != "persite" else np.array([])) > 0 and strategy != "persite":
        X_pool[:, global_mask] = 0.0

    masked_data[strategy] = {"train": tr, "test": te, "pooled_X": X_pool, "pooled_y": pooled_y_train}

print("Masked data variants created for all strategies.")

In [ ]:
# ── Model construction + helpers ──
# Thin torch glue shared by the centralized and federated phases:
# DatasetFromNumpy batches (X, y); build_mlp() instantiates the
# 6000 -> 512 -> (256, 128) -> 2 spectral MLP; model_to_numpy/numpy_to_model
# convert a state_dict to/from the list of ndarrays Flower transmits;
# mlp_predict_proba returns the resistant-class probability.

class DatasetFromNumpy(Dataset):
    """Minimal torch Dataset over a NumPy ``(X, y)`` pair.

    Wraps float32 spectra and integer labels for DataLoader batching by the
    centralized trainers, the Flower clients and the evaluation callbacks.
    """
    def __init__(self, X, y):
        """Store ``X``/``y`` as float32/long tensors."""
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        """Number of spectra in the dataset."""
        return len(self.X)
    def __getitem__(self, i):
        """Return the ``(spectrum, label)`` tensors at index ``i``."""
        return self.X[i], self.y[i]

def build_mlp(dropout_high):
    """Instantiate the binary spectral MLP (6000 -> 512 -> 256 -> 128 -> 2).

    Mirrors the architecture used in 04-07: attention disabled, and the
    pre-logit dropout is half of ``dropout_high``.

    Parameters
    ----------
    dropout_high : float
        Dropout after the hidden layers.

    Returns
    -------
    SpectralAttentionMLP
        Fresh model with a 2-class softmax head.
    """
    return SpectralAttentionMLP(
        input_dim=6000, n_classes=2, hidden_dim=512, head_dims=(256,128),
        dropout_high=dropout_high, dropout_low=dropout_high/2.0, use_attention=False)

def model_to_numpy(model):
    """Convert a model state_dict to CPU NumPy arrays."""
    return [v.cpu().numpy() for v in model.state_dict().values()]

def numpy_to_model(model, params):
    """Load a list of NumPy arrays (Flower parameters) into ``model``."""
    sd = model.state_dict()
    for k, p in zip(sd.keys(), params): sd[k] = torch.tensor(p)
    model.load_state_dict(sd)

def mlp_predict_proba(model, X_np, dev=DEV):
    """Resistant-class probability for each row of ``X_np`` (softmax[:, 1])."""
    model.eval()
    X_t = torch.tensor(X_np, dtype=torch.float32).to(dev)
    with torch.no_grad():
        return F.softmax(model(X_t), dim=1).cpu().numpy()[:, 1]

In [ ]:
# ── Centralized MLP grid search (unmasked, pooled data) ──
# 8 (lr) x 7 (dropout) grid on a 15% stratified holdout of the pooled
# unmasked train set; each combo trains 50 epochs and is scored by the best
# Balanced Accuracy over the threshold sweep. The winning lr/dropout/threshold
# are frozen and reused by every masked centralized MLP in Phase 3.
print("\n=== Centralized MLP Grid Search (unmasked) ===")
X_gs, X_gv, y_gs, y_gv = train_test_split(
    X_pool_unmasked, pooled_y_train, test_size=0.15, stratify=pooled_y_train, random_state=SEED)

best_ba, BEST_MLP_LR, BEST_MLP_DH, BEST_MLP_THRESH = -1.0, None, None, 0.5
def best_threshold(y_true, proba):
    """Pick the threshold maximising Balanced Accuracy on a probability vector.

    Ties are broken towards the threshold closest to 0.5, so a flat plateau
    keeps the most conservative cut.

    Parameters
    ----------
    y_true : ndarray (N,)
        Binary labels.
    proba : ndarray (N,)
        Positive-class probabilities.

    Returns
    -------
    best_t : float
        Threshold from ``THRESHOLDS`` with the highest Balanced Accuracy.
    best_ba : float
        The corresponding Balanced Accuracy.
    """
    best_t, best_ba = 0.5, -1.0
    for t in THRESHOLDS:
        ba = balanced_accuracy_score(y_true, proba >= t)
        if ba > best_ba + 1e-9 or (abs(ba - best_ba) <= 1e-9 and abs(t - 0.5) < abs(best_t - 0.5)):
            best_ba, best_t = ba, t
    return best_t, best_ba

grid_rows = []
for lr in LR_GRID:
    lr_best_ba, lr_best_d, lr_best_t = -1.0, None, 0.5
    lr_row = []
    for d in DROP_GRID:
        m = build_mlp(d).to(DEV)
        ds = DatasetFromNumpy(X_gs, y_gs); dl = DataLoader(ds, batch_size=64, shuffle=True)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-3)
        crit = nn.CrossEntropyLoss()
        for _ in range(50):
            m.train()
            for xb, yb in dl:
                if len(xb) <= 1: continue
                xb, yb = xb.to(DEV), yb.to(DEV)
                opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        proba = mlp_predict_proba(m, X_gv)
        t, ba = best_threshold(y_gv, proba)
        lr_row.append(ba)
        if ba > lr_best_ba:
            lr_best_ba, lr_best_d, lr_best_t = ba, d, t
        if ba > best_ba:
            best_ba, BEST_MLP_LR, BEST_MLP_DH, BEST_MLP_THRESH = ba, lr, d, t
    grid_rows.append(lr_row)
    print(f"  lr={lr:.1e}  best-drop={lr_best_d:.1f}  BA={lr_best_ba:.4f}  thresh={lr_best_t:.3f}")
print("\n  Grid (rows=lr, cols=drop):")
print(pd.DataFrame(grid_rows, index=[f"{lr:.1e}" for lr in LR_GRID],
                   columns=[f"{d:.2f}" for d in DROP_GRID]).to_string())
print(f"\nBest: lr={BEST_MLP_LR:.1e} drop={BEST_MLP_DH:.1f} thresh={BEST_MLP_THRESH:.3f} BA={best_ba:.4f}")

In [ ]:
# ── Train centralized MLP + RF on unmasked pooled ──
# Retrains the best grid configuration on the full pooled train set (warmup +
# cosine, best-validation checkpoint) and fits a GridSearchCV RF. Both are
# scored per site and on the combined pool; RF_PARAMS/BEST_RF_THRESH feed
# FedRF and the cross-site baselines.
print("\n=== Centralized MLP (unmasked) ===")
cent_m = build_mlp(BEST_MLP_DH).to(DEV)
ds = DatasetFromNumpy(X_pool_unmasked, pooled_y_train)
dl = DataLoader(ds, batch_size=64, shuffle=True)
opt = torch.optim.AdamW(cent_m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
crit = nn.CrossEntropyLoss(); best_loss, best_sd, patience = float("inf"), None, 0
for ep in range(100):
    cent_m.train()
    for xb, yb in dl:
        if len(xb) <= 1: continue
        xb, yb = xb.to(DEV), yb.to(DEV)
        if ep < 10:
            for pg in opt.param_groups: pg["lr"] = BEST_MLP_LR * (ep+1)/10
        opt.zero_grad(); crit(cent_m(xb), yb).backward(); opt.step()
    if ep >= 10: sched.step()
    if ep % 5 == 0:
        cent_m.eval()
        with torch.no_grad():
            vl = sum(crit(cent_m(xb.to(DEV)), yb.to(DEV)).item() for xb, yb in dl)/len(dl)
        if vl < best_loss: best_loss = vl; best_sd = {k:v.cpu().clone() for k,v in cent_m.state_dict().items()}; patience = 0
        else: patience += 1
        if patience >= 3: break
if best_sd: cent_m.load_state_dict(best_sd)
cent_m.eval()

centralized_mlp_results = {}
for site in SITE_ORDER:
    X_tt = client_test_unmasked[site][0]; y_tt = client_test_unmasked[site][1]
    proba = mlp_predict_proba(cent_m, X_tt)
    centralized_mlp_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    centralized_mlp_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = mlp_predict_proba(cent_m, combined_X_test_unmasked)
centralized_mlp_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= BEST_MLP_THRESH)
centralized_mlp_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
print(f"Centralized MLP (unmasked): All_BalAcc={centralized_mlp_results['All_BalAcc']:.4f}")

# Centralized RF
grid_rf = GridSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=1),
                        param_grid=RF_PARAM_GRID, cv=3, scoring="balanced_accuracy", n_jobs=-1)
n_total = len(pooled_y_train)
if n_total > RF_DS_MAX_SAMPLES:
    keep_rf, _ = train_test_split(np.arange(n_total), train_size=RF_DS_MAX_SAMPLES,
                                  stratify=pooled_y_train, random_state=SEED)
else:
    keep_rf = np.arange(n_total)
print(f"  RF tuning on {len(keep_rf)} stratified samples")
grid_rf.fit(X_pool_unmasked[keep_rf], pooled_y_train[keep_rf]); RF_PARAMS = grid_rf.best_params_
print(f"RF best: {RF_PARAMS}")
rf_cent = RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1).fit(X_pool_unmasked, pooled_y_train)

centralized_rf_results = {}
for site in SITE_ORDER:
    X_tt = client_test_unmasked[site][0]; y_tt = client_test_unmasked[site][1]
    proba = rf_cent.predict_proba(X_tt)[:,1]
    centralized_rf_results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= 0.5)
    centralized_rf_results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
proba_all = rf_cent.predict_proba(combined_X_test_unmasked)[:,1]
centralized_rf_results["All_BalAcc"] = balanced_accuracy_score(combined_y_test, proba_all >= 0.5)
centralized_rf_results["All_AUC"] = roc_auc_score(combined_y_test, proba_all)
print(f"Centralized RF (unmasked): All_BalAcc={centralized_rf_results['All_BalAcc']:.4f}")
BEST_RF_THRESH = 0.5; RF_TREES_PER_ROUND = RF_PARAMS["n_estimators"]
print(f"BEST_MLP_LR={BEST_MLP_LR:.2e} BEST_MLP_DH={BEST_MLP_DH:.1f} RF_TREES={RF_TREES_PER_ROUND}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 3: Mask comparison — Centralized MLP per mask
# ═══════════════════════════════════════════════════════════════════════════
# Centralized control for the mask comparison: one MLP per mask strategy
# (union/majority/persite) under the unmasked-tuned protocol, scored per site
# and pooled. Per-site deltas vs unmasked are printed; the results feed
# mask_comparison and the mask_delta figure.

print("\n=== Phase 3: Mask Comparison (Centralized MLP) ===")
mask_comparison = {"none": centralized_mlp_results}

for strategy in ["union", "majority", "persite"]:
    print(f"\n--- {strategy} mask ---")
    data = masked_data[strategy]
    X_pool = data["pooled_X"]

    m = build_mlp(BEST_MLP_DH).to(DEV)
    ds = DatasetFromNumpy(X_pool, data["pooled_y"])
    dl = DataLoader(ds, batch_size=64, shuffle=True)
    opt = torch.optim.AdamW(m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=90, eta_min=1e-6)
    crit = nn.CrossEntropyLoss()
    best_loss, best_sd, patience = float("inf"), None, 0
    for ep in range(100):
        m.train()
        for xb, yb in dl:
            if len(xb) <= 1: continue
            xb, yb = xb.to(DEV), yb.to(DEV)
            if ep < 10:
                for pg in opt.param_groups: pg["lr"] = BEST_MLP_LR * (ep+1)/10
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        if ep >= 10: sched.step()
        if ep % 5 == 0:
            m.eval()
            with torch.no_grad():
                vl = sum(crit(m(xb.to(DEV)), yb.to(DEV)).item() for xb, yb in dl)/len(dl)
            if vl < best_loss: best_loss = vl; best_sd = {k:v.cpu().clone() for k,v in m.state_dict().items()}; patience = 0
            else: patience += 1
            if patience >= 3: break
    if best_sd: m.load_state_dict(best_sd)
    m.eval()

    results = {}
    for site in SITE_ORDER:
        X_tt = data["test"][site][0]; y_tt = data["test"][site][1]
        proba = mlp_predict_proba(m, X_tt)
        results[f"{site}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
        results[f"{site}_AUC"] = roc_auc_score(y_tt, proba)
    combined_X = np.concatenate([data["test"][s][0] for s in SITE_ORDER])
    combined_y = np.concatenate([data["test"][s][1] for s in SITE_ORDER])
    proba_all = mlp_predict_proba(m, combined_X)
    results["All_BalAcc"] = balanced_accuracy_score(combined_y, proba_all >= BEST_MLP_THRESH)
    results["All_AUC"] = roc_auc_score(combined_y, proba_all)
    for site in SITE_ORDER:
        delta = results[f"{site}_BalAcc"] - centralized_mlp_results[f"{site}_BalAcc"]
        print(f"  {site}: masked={results[f'{site}_BalAcc']:.4f}  unmasked={centralized_mlp_results[f'{site}_BalAcc']:.4f}  Δ={delta:+.4f}")
    check_delta = results["All_BalAcc"] - centralized_mlp_results["All_BalAcc"]
    print(f"  All: masked={results['All_BalAcc']:.4f}  unmasked={centralized_mlp_results['All_BalAcc']:.4f}  Δ={check_delta:+.4f}")
    mask_comparison[strategy] = results

# Print summary table
print("\n  Mask Comparison Summary (Centralized MLP All_BalAcc):")
for s in MASK_STRATEGIES:
    print(f"    {s:10s}: {mask_comparison[s].get('All_BalAcc', np.nan):.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 3 continued: FedAvg MLP per mask
# ═══════════════════════════════════════════════════════════════════════════

# ── FL infra (train_local_mixed, FedMLPClient, etc.) ──
# The Flower building blocks reused by every MLP simulation: clients train
# n_mixes local passes (multi-mix surrogate for local epochs) with an optional
# FedProx proximal term, and get_mlp_eval_fn() records per-round per-site and
# pooled BalAcc/AUC plus a pooled cross-entropy val_loss. This cell then runs
# FedAvg once per mask strategy and keeps each round history.
def train_local_mixed(model, X_np, y_np, n_mixes_val, proximal_mu, global_params):
    """Run ``n_mixes_val`` local passes and return the mean training loss.

    Each pass reshuffles the local data with a mix-specific seed, trains one
    epoch of AdamW + cross-entropy, snapshots the weights, and the snapshots
    are averaged at the end -- a mixout-style surrogate for several local
    epochs. With ``proximal_mu > 0`` a FedProx proximal term additionally
    pulls the weights towards ``global_params``.

    Parameters
    ----------
    model : torch.nn.Module
        Client model, modified in place.
    X_np, y_np : ndarray
        Local training data.
    n_mixes_val : int
        Number of local passes.
    proximal_mu : float
        FedProx mu (0 disables the proximal term).
    global_params : list of torch.Tensor or None
        Global weights received from the server (required when mu > 0).

    Returns
    -------
    float
        Mean per-mix training loss across the local passes.
    """
    collect_sds = []; total_loss = 0.0
    for mix_i in range(n_mixes_val):
        seed = SEED + mix_i * 100 + int(proximal_mu * 1000)
        torch.manual_seed(seed); np.random.seed(seed)
        idx = np.random.permutation(len(X_np))
        X_shuf, y_shuf = X_np[idx], y_np[idx]
        ds = DatasetFromNumpy(X_shuf, y_shuf)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
        opt = torch.optim.AdamW(model.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        model.train(); batch_loss = 0.0; n_batch = 0
        for xb, yb in dl:
            if len(xb) <= 1: continue
            xb, yb = xb.to(FL_DEVICE), yb.to(FL_DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            if proximal_mu > 0 and global_params is not None:
                prox = sum((w - gw.to(FL_DEVICE)).norm(2) ** 2 for w, gw in zip(model.parameters(), global_params))
                loss = loss + (proximal_mu / 2.0) * prox
            loss.backward(); opt.step()
            batch_loss += loss.item(); n_batch += 1
        total_loss += batch_loss / n_batch
        collect_sds.append({k: v.cpu().clone() for k, v in model.state_dict().items()})
    if len(collect_sds) > 1:
        avg_sd = {}
        for k in collect_sds[0].keys():
            stacked = torch.stack([sd[k].float() for sd in collect_sds])
            avg_sd[k] = stacked.mean(0)
            if k.endswith("num_batches_tracked"):
                avg_sd[k] = avg_sd[k].to(torch.long) if avg_sd[k].dim()==0 else avg_sd[k].round().to(torch.long)
        model.load_state_dict(avg_sd)
    return total_loss / n_mixes_val

class FedMLPClient(fl.client.NumPyClient):
    """Flower NumPyClient wrapping one site's local MLP training.

    Parameters
    ----------
    cid : str
        Flower client id; ``SITE_ORDER[int(cid)]`` maps it to a DRIAMS site.
    X_train, y_train : ndarray
        The client's private (already preprocessed and masked) training data.
    """
    def __init__(self, cid, X_train, y_train):
        """Store the local data and build a fresh MLP on ``FL_DEVICE``."""
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        self.n_mixes_val = n_mixes(len(X_train))
        self.model = build_mlp(BEST_MLP_DH).to(FL_DEVICE)
    def get_parameters(self, config):
        """Return the current model weights as NumPy arrays."""
        return model_to_numpy(self.model)
    def set_parameters(self, params):
        """Load the server weights into the local model."""
        numpy_to_model(self.model, params)
    def fit(self, parameters, config):
        """Train locally and return weights, example count and metrics.

        Parameters
        ----------
        parameters : list of ndarray
            Global weights, installed before local training.
        config : dict
            Flower config; a positive ``proximal_mu`` enables the FedProx term.

        Returns
        -------
        tuple
            ``(updated weights, local #examples, {"train_loss", "n_mixes"})``.
        """
        self.set_parameters(parameters)
        proximal_mu = float(config.get("proximal_mu", 0.0))
        global_copy = None
        if proximal_mu > 0: global_copy = [p.clone().detach() for p in self.model.parameters()]
        loss = train_local_mixed(self.model, self.X_train, self.y_train,
                                 self.n_mixes_val, proximal_mu, global_copy)
        return (self.get_parameters({}), len(self.X_train),
                {"train_loss": loss, "n_mixes": self.n_mixes_val})

def get_mlp_eval_fn(test_dict, threshold, hist_list):
    """Build the server-side evaluation callback recorded per round.

    The callback instantiates a fresh MLP from the aggregated weights and
    appends one record per round: per-site BalAcc/AUC at the frozen threshold,
    pooled BalAcc/AUC, and a pooled cross-entropy ``val_loss``. The records
    drive the convergence and loss plots.

    Parameters
    ----------
    test_dict : dict
        ``{site: (X_test, y_test)}`` -- the masked test splits.
    threshold : float
        Decision threshold used for every Balanced Accuracy.
    hist_list : list
        Per-round accumulator, extended in place.

    Returns
    -------
    callable
        ``evaluate(server_round, parameters, config) -> (loss, metrics)``.
    """
    def evaluate(server_round, parameters, config):
        """Score the aggregated model for one round and record its metrics."""
        m = build_mlp(BEST_MLP_DH); numpy_to_model(m, parameters); m.to(FL_DEVICE); m.eval()
        record = {"round": server_round}
        ap, al = [], []
        ce_losses = []; ce_weights = []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            X_t = torch.tensor(X_tt, dtype=torch.float32).to(FL_DEVICE)
            y_t = torch.tensor(y_tt, dtype=torch.long).to(FL_DEVICE)
            with torch.no_grad():
                logits = m(X_t)
                loss = F.cross_entropy(logits, y_t).item()
            ce_losses.append(loss * len(y_tt)); ce_weights.append(len(y_tt))
            proba = F.softmax(logits, dim=1).cpu().numpy()[:, 1]
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        # val_loss = pooled-test cross-entropy; can rise while BalAcc rises (ranking vs calibration)
        record["val_loss"] = float(sum(ce_losses) / sum(ce_weights)) if sum(ce_weights) > 0 else 0.0
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

print("\n=== Phase 3 continued: FedAvg MLP per mask ===")
fedavg_hist_per_mask = {}
fedavg_train_hist_per_mask = {}
for strategy in MASK_STRATEGIES:
    print(f"\n--- {strategy} mask ---")
    data = masked_data[strategy]
    fedavg_hist = []; _eval_fn = get_mlp_eval_fn(data["test"], BEST_MLP_THRESH, fedavg_hist)
    fedavg_train_hist_per_mask[strategy] = []
    round_ctr = {"r": 0}
    def _fit_agg(metrics):
        """Aggregate client fit metrics and record the round mean train loss."""
        agg = weighted_avg_loss(metrics)
        round_ctr["r"] += 1
        fedavg_train_hist_per_mask[strategy].append({"round": round_ctr["r"], "All_train_loss": float(agg.get("avg_train_loss", np.nan))})
        return agg
    strategy_obj = fl.server.strategy.FedAvg(
        fraction_fit=1.0, fraction_evaluate=0.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, fit_metrics_aggregation_fn=_fit_agg,
        evaluate_fn=_eval_fn,
        initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp(BEST_MLP_DH))))

    def make_client_fn(train_dict):
        """Return a client_fn mapping a Flower cid to a FedMLPClient.

        Parameters
        ----------
        train_dict : dict
            ``{site: (X_train, y_train)}`` for the current mask strategy.

        Returns
        -------
        callable
            ``client_fn(cid) -> FedMLPClient`` used by the Flower simulation.
        """
        def _fn(cid):
            """Construct the FedMLPClient for ``cid`` from ``train_dict``."""
            site = SITE_ORDER[int(cid)]
            return FedMLPClient(cid, *train_dict[site]).to_client()
        return _fn

    fl.simulation.start_simulation(
        client_fn=make_client_fn(data["train"]), num_clients=4,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy_obj, client_resources={"num_cpus": 1, "num_gpus": 0})

    fedavg_hist_per_mask[strategy] = fedavg_hist.copy()
    last = fedavg_hist[-1] if fedavg_hist else {}
    print(f"  Final All_BalAcc: {last.get('All_BalAcc', np.nan):.4f}")
    print(f"  Best All_BalAcc:  {max((h.get('All_BalAcc',0) for h in fedavg_hist[1:]), default=0):.4f}")

In [ ]:
# ── Pick best mask: highest worst-site improvement ──
# Compares each mask's FedAvg peak per site against the unmasked FedAvg peak
# (rounds after the first) and keeps the strategy with the best worst-site
# delta -- the most conservative fair choice, as used by the published run.
print("\n=== Best Mask Selection ===")
best_strategy = "none"; best_worst = 0
for strategy in MASK_STRATEGIES:
    delta_per_site = []
    for site in SITE_ORDER:
        if strategy in fedavg_hist_per_mask and fedavg_hist_per_mask[strategy]:
            masked_val = max((h.get(f"{site}_BalAcc",0) for h in fedavg_hist_per_mask[strategy][1:]), default=0)
            unmasked_val = max((h.get(f"{site}_BalAcc",0) for h in fedavg_hist_per_mask["none"][1:]), default=0)
            delta_per_site.append(masked_val - unmasked_val)
    if delta_per_site:
        worst = min(delta_per_site)
        print(f"  {strategy:10s}: worst-site Δ={worst:+.4f}")
        if worst > best_worst:
            best_worst = worst; best_strategy = strategy

print(f"\nBest mask: {best_strategy} (worst-site improvement: {best_worst:+.4f})")

In [ ]:
# ── Switch active data to best mask ──
# Rebinds client_train_pp/client_test_pp to the winning mask's arrays so the
# full FL pipeline in Phase 4 and the cross-site baselines below train on
# exactly the same masked representation.
client_train_pp = masked_data[best_strategy]["train"]
client_test_pp = masked_data[best_strategy]["test"]
X_best_pool = masked_data[best_strategy]["pooled_X"]
print(f"Switched to best mask: {best_strategy}")
print(f"  Masked bins: ", end="")
if best_strategy == "union": print(f"{len(union_mask)} (union)")
elif best_strategy == "majority": print(f"{len(majority_mask)} (majority)")
elif best_strategy == "persite": print(f"~{MASK_TOP_K} per site (persite)")
else: print("none (unmasked)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 4: Full FL pipeline with best mask
# ═══════════════════════════════════════════════════════════════════════════
# Full FL pipeline on the winning mask: FedProx runs per mu, the Phase-3
# FedAvg history is reused, and FedLR/FedRF close the method set before the
# A->B/C/D cross-site baselines, result table and figures.

In [ ]:
# Reuses the FedAvg MLP history collected for the winning mask in Phase 3
# instead of re-running it (identical configuration), so all strategies
# plotted and exported below share the same mask.
fedavg_hist = fedavg_hist_per_mask[best_strategy]
print(f"FedAvg MLP ({best_strategy} mask): {len(fedavg_hist)} rounds available from Phase 3")

In [ ]:
# ── FedProx MLP ──
# CheckpointFedProx wraps Flower's FedProx to save client/global weights each
# round and to log per-site + mean training loss; one simulation per mu from
# FEDPROX_MUS. The proximal term penalises client-weight divergence from the
# global model (FedProx), matching the docs' mu notation.
class CheckpointFedProx(fl.server.strategy.FedProx):
    """FedProx strategy that checkpoints models and logs training losses.

    Extends Flower's ``FedProx`` so every round dumps each client's weights and
    the aggregated global model to ``models/<strategy>/round_XXX/``, and appends
    a per-site plus mean training-loss record to ``train_hist_ref``.
    """
    def __init__(self, strategy_name, model_dir_path, train_hist_ref, **kw):
        """Set the checkpoint directory and the training-history accumulator.

        Parameters
        ----------
        strategy_name : str
            Names the subdirectory under ``model_dir_path``.
        model_dir_path : str or Path
            Root models directory (the notebook passes ``MODEL_DIR``).
        train_hist_ref : list
            Extended with one ``{"round", "<site>_train_loss", ...,
            "All_train_loss"}`` dict per round.
        kw : dict
            Forwarded to ``fl.server.strategy.FedProx`` (fraction_fit,
            proximal_mu, evaluate_fn, ...).
        """
        super().__init__(**kw)
        self.strategy_name = strategy_name
        self.model_dir_path = Path(model_dir_path) / strategy_name
        self.model_dir_path.mkdir(parents=True, exist_ok=True)
        self.train_hist_ref = train_hist_ref
    def aggregate_fit(self, server_round, results, failures):
        """Checkpoint client and global weights, record losses, then aggregate.

        Parameters
        ----------
        server_round : int
            Current federated round.
        results : list
            ``(ClientProxy, FitRes)`` pairs from this round's clients.
        failures : list
            Failed client results, forwarded to the parent implementation.

        Returns
        -------
        tuple
            ``(aggregated parameters, metrics)`` exactly as FedProx returns them.
        """
        round_dir = self.model_dir_path / f"round_{server_round:03d}"; round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            m = build_mlp(BEST_MLP_DH); numpy_to_model(m, ndarrays)
            torch.save(m.state_dict(), round_dir / f"client_{cp.cid}.pt")
        train_record = {"round": server_round}; train_losses = []
        for cp, fit_res in results:
            cid_idx = int(cp.cid); site = SITE_ORDER[cid_idx] if cid_idx < len(SITE_ORDER) else f"site_{cp.cid}"
            tl = fit_res.metrics.get("train_loss", float("nan"))
            train_record[f"{site}_train_loss"] = float(tl)
            train_losses.append(float(tl))
        train_record["All_train_loss"] = float(np.mean(train_losses)) if train_losses else float("nan")
        self.train_hist_ref.append(train_record)
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            gm = build_mlp(BEST_MLP_DH); numpy_to_model(gm, nd)
            torch.save(gm.state_dict(), round_dir / "global_model.pt")
        return aggregated, metrics

fedprox_histories = {}
fedprox_train_histories = {}
for mu in FEDPROX_MUS:
    print(f"\n=== FedProx MLP (mu={mu}, {best_strategy} mask) ===")
    fedprox_hist = []; _eval_fn = get_mlp_eval_fn(client_test_pp, BEST_MLP_THRESH, fedprox_hist)
    fedprox_train_hist = []
    s = CheckpointFedProx(f"fedprox_mlp_mu{mu}_{best_strategy}", MODEL_DIR, fedprox_train_hist,
        fraction_fit=1.0, fraction_evaluate=0.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, fit_metrics_aggregation_fn=weighted_avg_loss,
        proximal_mu=mu, evaluate_fn=_eval_fn,
        initial_parameters=fl.common.ndarrays_to_parameters(model_to_numpy(build_mlp(BEST_MLP_DH))))
    s.model_dir_path = Path(MODEL_DIR) / f"fedprox_mlp_mu{mu}_{best_strategy}"

    def make_client_fn(train_dict):
        """Map a Flower cid to the FedProx client for its DRIAMS site.

        Parameters
        ----------
        train_dict : dict
            ``{site: (X_train, y_train)}`` -- here the winning mask's arrays.

        Returns
        -------
        callable
            ``client_fn(cid) -> FedMLPClient.to_client()``.
        """
        def _fn(cid):
            """Construct the client for the site encoded by ``cid``."""
            site = SITE_ORDER[int(cid)]
            return FedMLPClient(cid, *train_dict[site]).to_client()
        return _fn

    fl.simulation.start_simulation(
        client_fn=make_client_fn(client_train_pp), num_clients=4,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=s, client_resources={"num_cpus": 1, "num_gpus": 0})
    fedprox_histories[mu] = fedprox_hist.copy()
    fedprox_train_histories[mu] = fedprox_train_hist.copy()
    print(f"FedProx mu={mu} done. {len(fedprox_hist)} rounds.")

In [ ]:
# ── FedLR with tuned C ──
# Federated logistic regression: per-site 3-fold CV picks C (worst-site rule)
# and the decision threshold, then warm-started FedAvg rounds aggregate
# coef_/intercept_. The evaluate callback records per-round site metrics and
# the local log-loss as the training loss.
print("\n=== FedAvg LR (tuned C) ===")
C_GRID = np.linspace(5e-5, 1e-3, 15)
site_lr_scores = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    grid = GridSearchCV(LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                                             max_iter=5000, random_state=SEED),
                        param_grid={"C": C_GRID}, cv=3, scoring="balanced_accuracy", n_jobs=-1)
    grid.fit(X_tr, y_tr)
    for i, params in enumerate(grid.cv_results_["params"]):
        key = float(params["C"])
        score = grid.cv_results_["mean_test_score"][i]
        site_lr_scores.setdefault(key, []).append(score)
BEST_LR_C = max(site_lr_scores, key=lambda c: min(site_lr_scores[c]))
print(f"Tuned LR C (worst-site): {BEST_LR_C:.2e}")

# Per-site LR threshold tuning (worst-site optimal)
lr_thresh_scores = {t: [] for t in THRESHOLDS}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    cv_proba = cross_val_predict(
        LogisticRegression(C=BEST_LR_C, penalty="l2", solver="lbfgs",
                           class_weight="balanced", max_iter=5000, random_state=SEED),
        X_tr, y_tr, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    for t in THRESHOLDS:
        lr_thresh_scores[t].append(balanced_accuracy_score(y_tr, cv_proba >= t))
BEST_LR_THRESH = max(THRESHOLDS, key=lambda t: min(lr_thresh_scores[t]))
print(f"Tuned LR threshold (worst-site): {BEST_LR_THRESH:.3f}")

class FedLRClient(fl.client.NumPyClient):
    """Flower NumPyClient wrapping one site's warm-started logistic regression.

    The model is a SAGA ``LogisticRegression`` kept at ``max_iter=1`` with
    ``warm_start=True``: every ``fit()`` performs one pass from the server's
    coef_/intercept_, so the FedAvg rounds compose into federated training.
    """
    def __init__(self, cid, X_train, y_train):
        """Initialise a zero-coefficient LR at the notebook-tuned C."""
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        nf = X_train.shape[1]
        self.model = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga", max_iter=1,
                                        warm_start=True, class_weight="balanced", random_state=SEED)
        self.model.classes_ = np.array([0,1]); self.model.coef_ = np.zeros((1,nf)); self.model.intercept_ = np.zeros(1)
    def get_parameters(self, config):
        """Return ``[coef_, intercept_]`` as the two FedAvg arrays."""
        return [self.model.coef_.ravel(), self.model.intercept_]
    def set_parameters(self, params):
        """Load the server coefficient and intercept arrays."""
        self.model.coef_ = params[0].reshape(1,-1); self.model.intercept_ = params[1]
    def fit(self, parameters, config):
        """Run one warm-started SAGA pass on the local data.

        Returns
        -------
        tuple
            ``(updated [coef_, intercept_], local #examples, {"train_loss",
            "num_examples"})`` with ``train_loss`` the local LR log-loss.
        """
        self.set_parameters(parameters)
        with warnings.catch_warnings(): warnings.simplefilter("ignore"); self.model.fit(self.X_train, self.y_train)
        tl = log_loss(self.y_train, self.model.predict_proba(self.X_train)[:, 1], labels=[0, 1])
        return (self.get_parameters({}), len(self.X_train), {"train_loss": float(tl), "num_examples": len(self.X_train)})

def get_lr_eval_fn(test_dict, hist_list):
    """Build the per-round evaluation callback for the federated LR.

    Re-instantiates an LR from the server arrays and records per-site
    BalAcc/AUC at the tuned threshold plus the pooled values in ``hist_list``.

    Parameters
    ----------
    test_dict : dict
        ``{site: (X_test, y_test)}`` (winning-mask test splits).
    hist_list : list
        Per-round accumulator, extended in place.

    Returns
    -------
    callable
        ``evaluate(server_round, parameters, config) -> (loss, metrics)``.
    """
    def evaluate(server_round, parameters, config):
        """Score the federated LR for one round and append its record."""
        lr = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="lbfgs", class_weight="balanced")
        lr.classes_ = np.array([0,1]); lr.coef_ = parameters[0].reshape(1,-1); lr.intercept_ = parameters[1]
        record = {"round": server_round}; ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = lr.predict_proba(X_tt)[:,1]
            preds = proba >= BEST_LR_THRESH
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= BEST_LR_THRESH))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

fedlr_hist = []; _eval_fn = get_lr_eval_fn(client_test_pp, fedlr_hist)
fedlr_train_hist = []
lr_round_ctr = {"r": 0}
def lr_fit_agg(metrics):
    """Aggregate client LR metrics and record the round mean log-loss."""
    agg = weighted_avg_loss(metrics)
    lr_round_ctr["r"] += 1
    fedlr_train_hist.append({"round": lr_round_ctr["r"], "All_train_loss": float(agg.get("avg_train_loss", np.nan))})
    return agg
n_feat = client_train_pp[SITE_ORDER[0]][0].shape[1]
lr_init = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga", max_iter=1,
                              warm_start=True, class_weight="balanced", random_state=SEED)
lr_init.classes_ = np.array([0,1]); lr_init.coef_ = np.zeros((1,n_feat)); lr_init.intercept_ = np.zeros(1)
lr_strat = fl.server.strategy.FedAvg(
    fraction_fit=1.0, fraction_evaluate=0.0,
    min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, fit_metrics_aggregation_fn=lr_fit_agg, evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters([lr_init.coef_.ravel(), lr_init.intercept_]))

def lr_client_fn(cid):
    """Build the FedLRClient for the site encoded by ``cid``."""
    site = SITE_ORDER[int(cid)]
    return FedLRClient(cid, *client_train_pp[site]).to_client()

fl.simulation.start_simulation(client_fn=lr_client_fn, num_clients=4,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=lr_strat, client_resources={"num_cpus": 1, "num_gpus": 0})
print(f"FedLR done. {len(fedlr_hist)} rounds.")

In [ ]:
# ── FedRF tree collection ──
# Each round every client trains a fresh local RF; the server stores the
# union of all clients' trees (tree collection) and re-serialises the
# ensemble for evaluation. The RF decision threshold is tuned worst-site by
# 3-fold CV first, because tree-collection probabilities can otherwise be
# stuck at 0.5.
def trees_to_array(est):
    """Serialise a list of fitted trees to a uint8 joblib blob.

    Parameters
    ----------
    est : list of DecisionTreeClassifier
        The client's locally fitted trees.

    Returns
    -------
    ndarray of uint8
        Bytes that Flower can ship as a single parameter array.
    """
    buf = io.BytesIO(); joblib.dump(est, buf); buf.seek(0)
    return np.frombuffer(buf.read(), dtype=np.uint8)
def array_to_trees(arr):
    """Deserialise a joblib blob back into a list of fitted trees."""
    return joblib.load(io.BytesIO(arr.tobytes()))

class TreeCollectionFedAvg(fl.server.strategy.FedAvg):
    """FedAvg variant that aggregates Random Forest trees by collection.

    Instead of averaging weights, the server appends every client's trees to an
    accumulated forest and ships the whole ensemble back, so the global model
    grows linearly with clients x rounds (tree collection).
    """
    def __init__(self, **kw):
        """Initialise the accumulated (global) tree list."""
        super().__init__(**kw); self._accumulated_trees = []
    def aggregate_fit(self, server_round, results, failures):
        """Collect this round's trees and return the accumulated ensemble.

        Parameters
        ----------
        server_round : int
            Current federated round.
        results : list
            ``(ClientProxy, FitRes)`` pairs; each carries one joblib blob of trees.
        failures : list
            Unused, kept for the Flower strategy signature.

        Returns
        -------
        tuple
            ``(parameters, {"total_trees": int})`` with ``parameters`` the joblib
            blob of the accumulated forest (``None`` when no client reported).
        """
        if not results: return None, {}
        for _, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            if len(ndarrays[0]) > 0: self._accumulated_trees.extend(array_to_trees(ndarrays[0]))
        combined = trees_to_array(self._accumulated_trees)
        aggregated = fl.common.ndarrays_to_parameters([combined])
        metrics = {"total_trees": len(self._accumulated_trees)}
        return aggregated, metrics

class FedRFClient(fl.client.NumPyClient):
    """Flower NumPyClient that trains a fresh local Random Forest per round.

    The client owns no global weights: it only contributes trees, and the server
    keeps the accumulated ensemble (see ``TreeCollectionFedAvg``).
    """
    def __init__(self, cid, X_train, y_train):
        """Store the site's local training data."""
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
    def get_parameters(self, config):
        """Return an empty array -- clients do not own global weights."""
        return [np.array([], dtype=np.uint8)]
    def fit(self, parameters, config):
        """Fit a round-seeded RF on the local data and return its trees.

        The seed mixes the client id and the server round, so every round adds a
        different ensemble; ``RF_PARAMS`` is the grid-search winner from the
        centralized baseline.

        Returns
        -------
        tuple
            ``(joblib blob of trees, local #examples, {"n_new_trees",
            "num_examples"})``.
        """
        server_round = int(config.get("current_round", 0))
        rf = RandomForestClassifier(**RF_PARAMS, random_state=SEED+int(self.cid)+100*server_round,
                                     n_jobs=-1, warm_start=True)
        rf.fit(self.X_train, self.y_train)
        return ([trees_to_array(list(rf.estimators_))], len(self.X_train),
                {"n_new_trees": len(rf.estimators_), "num_examples": len(self.X_train)})

def get_rf_eval_fn(test_dict, hist_list):
    """Build the per-round evaluation callback for FedRF.

    Deserialises the accumulated forest and scores the pooled + per-site
    BalAcc/AUC at the tuned RF threshold, appending a record to ``hist_list``.
    The first round (no trees yet) returns a sentinel 0.5 record.

    Parameters
    ----------
    test_dict : dict
        ``{site: (X_test, y_test)}`` (winning-mask test splits).
    hist_list : list
        Per-round accumulator, extended in place.

    Returns
    -------
    callable
        ``evaluate(server_round, parameters, config) -> (loss, metrics)``.
    """
    def evaluate(server_round, parameters, config):
        """Score the accumulated forest for one round and append its record."""
        if len(parameters[0]) == 0: return (1.0, {"All_BalAcc": 0.5})
        trees = array_to_trees(parameters[0])
        rf = RandomForestClassifier(**RF_PARAMS, n_jobs=-1); rf.estimators_ = trees
        rf.n_classes_ = 2; rf.classes_ = np.array([0,1]); rf.n_outputs_ = 1
        record = {"round": server_round, "n_trees": len(trees)}; ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = client_test_pp[site]
            proba = rf.predict_proba(X_tt)[:,1]
            preds = proba >= BEST_RF_THRESH
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= BEST_RF_THRESH))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

print("\n=== FedRF (Tree Collection) ===")
# Tune RF decision threshold (worst-site optimal) so tree-collection probabilities are not stuck at 0.5
rf_thresh_scores = {t: [] for t in THRESHOLDS}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    cv_proba = cross_val_predict(RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1),
                                 X_tr, y_tr, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    for t in THRESHOLDS:
        rf_thresh_scores[t].append(balanced_accuracy_score(y_tr, cv_proba >= t))
BEST_RF_THRESH = max(THRESHOLDS, key=lambda t: min(rf_thresh_scores[t]))
print(f"Tuned RF threshold (worst-site): {BEST_RF_THRESH:.3f}")
fedrf_hist = []; _eval_fn = get_rf_eval_fn(client_test_pp, fedrf_hist)
rf_strat = TreeCollectionFedAvg(
    fraction_fit=1.0, fraction_evaluate=0.0,
    min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, fit_metrics_aggregation_fn=weighted_avg_loss, evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters([np.array([], dtype=np.uint8)]))

def rf_client_fn(cid):
    """Build the FedRFClient for the site encoded by ``cid``."""
    site = SITE_ORDER[int(cid)]
    return FedRFClient(cid, *client_train_pp[site]).to_client()

fl.simulation.start_simulation(client_fn=rf_client_fn, num_clients=4,
    config=fl.server.ServerConfig(num_rounds=NUM_RF_ROUNDS),
    strategy=rf_strat, client_resources={"num_cpus": 1, "num_gpus": 0})
fedrf_hist = fedrf_hist.copy()  # noqa F841
print(f"FedRF done. {len(fedrf_hist)} rounds.")

In [ ]:
# ── Cross-Site MLP + RF (best mask, persite-aware) ──
# Non-federated lower bound: train on site A only (log1p+standardize fitted
# on A, then the winning mask zeroed -- each test site's own top-500 when the
# strategy is persite) and evaluate on B/C/D plus their pool. Contrasting
# this gap with the federated numbers shows what collaboration buys.
print("\n=== Cross-Site MLP + RF (A -> B/C/D) ===")
X_A_train, y_A_train = client_train["A"]
state_cs = fit_input_transform(X_A_train, "log1p+standardize")
X_A_tr = apply_input_transform(X_A_train, state_cs)

# Apply best mask (persite-aware)
mask_A = np.array([], dtype=int)
if best_strategy == "union":
    mask_A = union_mask
elif best_strategy == "majority":
    mask_A = majority_mask
elif best_strategy == "persite":
    mask_A = site_masks_raw["A"] if len(site_masks_raw["A"]) > 0 else np.array([], dtype=int)
if len(mask_A) > 0:
    X_A_tr[:, mask_A] = 0.0

cross_test_sets = {}
for sk in "BCD":
    X_te, y_te = client_test[sk]
    X_te_pp = apply_input_transform(X_te, state_cs)
    # Per-site mask
    if best_strategy == "union":
        X_te_pp[:, union_mask] = 0.0
    elif best_strategy == "majority":
        X_te_pp[:, majority_mask] = 0.0
    elif best_strategy == "persite":
        mask_site = site_masks_raw[sk] if len(site_masks_raw[sk]) > 0 else np.array([], dtype=int)
        if len(mask_site) > 0: X_te_pp[:, mask_site] = 0.0
    cross_test_sets[sk] = (X_te_pp, y_te)
# A-val
X_te_pp = apply_input_transform(client_test["A"][0], state_cs)
if len(mask_A) > 0: X_te_pp[:, mask_A] = 0.0
cross_test_sets["A"] = (X_te_pp, client_test["A"][1])

# ── Cross-site MLP ──
cs_m = build_mlp(BEST_MLP_DH).to(DEV)
ds_cs = DatasetFromNumpy(X_A_tr, y_A_train)
dl_cs = DataLoader(ds_cs, batch_size=64, shuffle=True)
opt_cs = torch.optim.AdamW(cs_m.parameters(), lr=BEST_MLP_LR, weight_decay=1e-4)
sched_cs = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cs, T_max=90, eta_min=1e-6)
crit_cs = nn.CrossEntropyLoss()
for ep in range(100):
    cs_m.train()
    for xb, yb in dl_cs:
        if len(xb) <= 1: continue
        xb, yb = xb.to(DEV), yb.to(DEV)
        if ep < 10:
            for pg in opt_cs.param_groups: pg["lr"] = BEST_MLP_LR * (ep+1)/10
        opt_cs.zero_grad(); crit_cs(cs_m(xb), yb).backward(); opt_cs.step()
    if ep >= 10: sched_cs.step()
cs_m.eval()
cross_site_results = {}
for name, (X_tt, y_tt) in cross_test_sets.items():
    proba = mlp_predict_proba(cs_m, X_tt)
    cross_site_results[f"{name}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_MLP_THRESH)
    cross_site_results[f"{name}_AUC"] = roc_auc_score(y_tt, proba)
    print(f"  CS MLP {name}: BalAcc={cross_site_results[f'{name}_BalAcc']:.4f}  AUC={cross_site_results[f'{name}_AUC']:.4f}")
X_BCD = np.concatenate([cross_test_sets[s][0] for s in "BCD"])
y_BCD = np.concatenate([cross_test_sets[s][1] for s in "BCD"])
proba_bcd = mlp_predict_proba(cs_m, X_BCD)
cross_site_results["All_BalAcc"] = balanced_accuracy_score(y_BCD, proba_bcd >= BEST_MLP_THRESH)
cross_site_results["All_AUC"] = roc_auc_score(y_BCD, proba_bcd)
print(f"  CS MLP All (B+C+D): BalAcc={cross_site_results['All_BalAcc']:.4f}  AUC={cross_site_results['All_AUC']:.4f}")

# ── Cross-site RF ──
print("\n=== Cross-Site RF (A -> B/C/D) ===")
rf_cs = RandomForestClassifier(**RF_PARAMS, random_state=SEED, n_jobs=-1)
rf_cs.fit(X_A_tr, y_A_train)
cross_site_rf_results = {}
for name, (X_tt, y_tt) in cross_test_sets.items():
    proba = rf_cs.predict_proba(X_tt)[:, 1]
    cross_site_rf_results[f"{name}_BalAcc"] = balanced_accuracy_score(y_tt, proba >= BEST_RF_THRESH)
    cross_site_rf_results[f"{name}_AUC"] = roc_auc_score(y_tt, proba)
    print(f"  CS RF  {name}: BalAcc={cross_site_rf_results[f'{name}_BalAcc']:.4f}  AUC={cross_site_rf_results[f'{name}_AUC']:.4f}")
proba_bcd_rf = rf_cs.predict_proba(X_BCD)[:, 1]
cross_site_rf_results["All_BalAcc"] = balanced_accuracy_score(y_BCD, proba_bcd_rf >= BEST_RF_THRESH)
cross_site_rf_results["All_AUC"] = roc_auc_score(y_BCD, proba_bcd_rf)
print(f"  CS RF  All (B+C+D): BalAcc={cross_site_rf_results['All_BalAcc']:.4f}  AUC={cross_site_rf_results['All_AUC']:.4f}")

In [ ]:
# ── Assemble final results DataFrame ──
# Collects one row per method: centralized MLPs/RF, every mask variant, the
# FL strategies at their best pooled round (peak round; round 0 skipped) and
# the cross-site baselines. df_results drives the plots, heatmaps and CSVs.
def best_metrics(h):
    """Best pooled-BalAcc round from a per-round history, skipping round 0.

    Parameters
    ----------
    h : list of dict
        Per-round records; round 0 is the initial/untrained evaluation.

    Returns
    -------
    best : dict
        Record with the highest ``All_BalAcc`` (``{}`` for an empty history).
    peak_round : int
        ``round`` of that record (0 when the history is empty).
    """
    if not h: return {}, 0
    skip0 = h[1:]
    best = max(skip0, key=lambda r: r.get("All_BalAcc", 0))
    return best, int(best.get("round", 0))

rows = []
def add_row(method, cs_source=None, fed_hist=None):
    """Append one method's metrics to the result rows.

    Centralized/cross-site sources are read directly (Peak_Round 0); federated
    histories contribute their best pooled round, with the round shown in the
    row label.

    Parameters
    ----------
    method : str
        Display name, e.g. ``"FL FedProx mu=0.5 (none)"``.
    cs_source : dict or None
        Pre-computed ``{site}_BalAcc/{site}_AUC/All_*`` metrics.
    fed_hist : list of dict or None
        Per-round federated history (mutually exclusive with ``cs_source``).
    """
    r = {"Method": method}; peak_round = ""
    for site in SITE_ORDER:
        if cs_source:
            r[f"{site}_BalAcc"] = cs_source.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = cs_source.get(f"{site}_AUC", np.nan)
        elif fed_hist is not None:
            lm, pr = best_metrics(fed_hist)
            r[f"{site}_BalAcc"] = lm.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = lm.get(f"{site}_AUC", np.nan)
            peak_round = f" (r{pr})"
    if fed_hist is not None:
        lm, pr = best_metrics(fed_hist)
        r["All_BalAcc"] = lm.get("All_BalAcc", np.nan); r["All_AUC"] = lm.get("All_AUC", np.nan)
        r["Peak_Round"] = int(pr)
    elif cs_source:
        r["All_BalAcc"] = cs_source.get("All_BalAcc", np.nan); r["All_AUC"] = cs_source.get("All_AUC", np.nan)
        r["Peak_Round"] = 0
    r["Label"] = method + peak_round; rows.append(r)

add_row("Centralized MLP (unmasked)", cs_source=centralized_mlp_results)
add_row("Centralized RF (unmasked)", cs_source=centralized_rf_results)
for strategy in MASK_STRATEGIES:
    add_row(f"Centralized MLP ({strategy})", cs_source=mask_comparison[strategy])
if "fedavg_hist" in dir() and fedavg_hist:
    add_row(f"FL FedAvg MLP ({best_strategy})", fed_hist=fedavg_hist)
for mu in FEDPROX_MUS:
    if mu in fedprox_histories and fedprox_histories[mu]:
        add_row(f"FL FedProx mu={mu} ({best_strategy})", fed_hist=fedprox_histories[mu])
if "fedlr_hist" in dir(): add_row("FL FedAvg LR", fed_hist=fedlr_hist)
if "fedrf_hist" in dir(): add_row("FL FedRF (Trees)", fed_hist=fedrf_hist)
add_row("Cross-Site MLP", cs_source=cross_site_results)
add_row("Cross-Site RF", cs_source=cross_site_rf_results)

df_results = pd.DataFrame(rows)
cols = ["Method", "Label", "Peak_Round"] + [f"{s}_BalAcc" for s in SITE_ORDER] + ["All_BalAcc"]
print(df_results[cols].to_string(index=False))

In [ ]:
# ── Convergence plot ──
# Three panels written to results/convergence.pdf: FedAvg pooled BalAcc per
# mask (the Phase-3 comparison), all FL methods under the winning mask with
# the centralized MLP as reference, and their pooled cross-entropy val_loss.
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
ax = axes[0]
for strategy in MASK_STRATEGIES:
    h = fedavg_hist_per_mask[strategy]
    vals = [r.get("All_BalAcc", np.nan) for r in h]
    ax.plot(range(1, len(vals)+1), vals, label=f"{strategy} mask")
ax.set_title(f"FedAvg MLP — Mask Comparison"); ax.set_xlabel("Round"); ax.set_ylabel("All_BalAcc")
ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

ax = axes[1]
plots = [("FedAvg MLP", fedavg_hist, "#ff7f0e", "-")]
for mu in FEDPROX_MUS:
    if mu in fedprox_histories: plots.append((f"FedProx mu={mu}", fedprox_histories[mu], "#d62728", "--"))
plots.append(("FedAvg LR", fedlr_hist, "#1f77b4", "-."))
plots.append(("FedRF", fedrf_hist, "#2ca02c", "-"))
for label, hist, c, ls in plots:
    vals = [h.get("All_BalAcc", np.nan) for h in hist]
    ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=2, label=label)
ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=2, label='Centralized MLP (unmasked)')
ax.set_title(f"All Methods ({best_strategy} mask)"); ax.set_xlabel("Round"); ax.set_ylabel("All_BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

ax = axes[2]
for label, hist, c, ls in plots:
    vals = [h.get("val_loss", np.nan) for h in hist if "val_loss" in h]
    if vals: ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=1.5, label=label)
ax.set_title(f"Validation Loss ({best_strategy} mask)"); ax.set_xlabel("Round"); ax.set_ylabel("CE Loss")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5)

fig.suptitle(f"{DRUG_NAME} — Species-Masked Federated Learning", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "convergence.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# ── Loss curves: training vs validation ──
# Per-round training loss (client-weighted for FedAvg/FedProx, log-loss for
# FedLR) against the pooled validation cross-entropy, saved as
# results/loss_curves.pdf.
# Note: train/val loss (CE) can rise while balanced accuracy rises — ranking vs calibration.
loss_methods = [("FedAvg MLP", fedavg_train_hist_per_mask.get(best_strategy, []), fedavg_hist)]
for mu in FEDPROX_MUS:
    loss_methods.append((f"FedProx mu={mu}", fedprox_train_histories.get(mu, []), fedprox_histories.get(mu, [])))
loss_methods.append(("FedAvg LR", fedlr_train_hist, fedlr_hist))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for label, th, vh in loss_methods:
    vals = [r.get("All_train_loss", np.nan) for r in th]
    if vals: ax.plot(range(1, len(vals)+1), vals, lw=2, label=label)
ax.set_title("Training Loss"); ax.set_xlabel("Round"); ax.set_ylabel("CE / log loss")
ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.5)

ax = axes[1]
for label, th, vh in loss_methods:
    vals = [r.get("val_loss", np.nan) for r in vh]
    if vals: ax.plot(range(1, len(vals)+1), vals, lw=2, label=label)
ax.set_title("Validation Loss"); ax.set_xlabel("Round"); ax.set_ylabel("CE loss")
ax.legend(fontsize=8); ax.grid(True, ls='--', alpha=0.5)

fig.suptitle(f"{DRUG_NAME} — Training vs Validation Loss ({best_strategy} mask)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "loss_curves.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Heatmaps of per-method/per-site Balanced Accuracy and AUC-ROC (seaborn,
# RdYlGn 0.4-1.0) written as results/heatmap_balacc.pdf and heatmap_auc.pdf;
# one row per method label from df_results with an 'All' pooled column.
ba_data = {}
for _, r in df_results.iterrows():
    ba_data[r["Label"]] = {f"Site {s}": r[f"{s}_BalAcc"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_BalAcc", np.nan)): ba_data[r["Label"]]["All"] = r["All_BalAcc"]
df_ba = pd.DataFrame(ba_data).T
df_ba = df_ba[[c for c in [f"Site {s}" for s in SITE_ORDER]+["All"] if c in df_ba.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_ba)*0.5)))
sns.heatmap(df_ba, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white", cbar_kws={"label":"Balanced Accuracy"}, ax=ax)
ax.set_title(f"{DRUG_NAME} — Species-Masked BalAcc ({best_strategy})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_balacc.pdf", bbox_inches="tight"); plt.show()

auc_data = {}
for _, r in df_results.iterrows():
    auc_data[r["Label"]] = {f"Site {s}": r[f"{s}_AUC"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_AUC", np.nan)): auc_data[r["Label"]]["All"] = r["All_AUC"]
df_auc = pd.DataFrame(auc_data).T
df_auc = df_auc[[c for c in [f"Site {s}" for s in SITE_ORDER]+["All"] if c in df_auc.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_auc)*0.5)))
sns.heatmap(df_auc, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white", cbar_kws={"label":"AUC-ROC"}, ax=ax)
ax.set_title(f"{DRUG_NAME} — Species-Masked AUC ({best_strategy})", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "heatmap_auc.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# ── Mask Δ vs unmasked ──
# Per-site Balanced Accuracy change of each mask strategy relative to the
# unmasked centralized MLP (masked - unmasked), saved as
# results/mask_delta.pdf; in the published run masking helped site B only.
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(SITE_ORDER)); w = 0.2
for i, strategy in enumerate(["union", "majority", "persite"]):
    deltas = [mask_comparison[strategy].get(f"{site}_BalAcc", 0) - mask_comparison["none"].get(f"{site}_BalAcc", 0)
              for site in SITE_ORDER]
    ax.bar(x + (i-1)*w, deltas, w, label=strategy)
ax.set_xticks(x); ax.set_xticklabels([f"Site {s}" for s in SITE_ORDER])
ax.axhline(0, color='black', lw=1)
ax.set_ylabel("Δ BalAcc (masked - unmasked)"); ax.set_title("Species Masking Effect per Site (Centralized MLP)")
ax.legend(); ax.grid(True, ls='--', alpha=0.5, axis='y')
plt.tight_layout(); plt.savefig(OUT_DIR / "mask_delta.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# Exports the per-round histories and final table to results/*.csv, writes
# best_params_used.txt (drug, best mask, lr/dropout/threshold, LR C, RF
# params) and archives this notebook into results/notebook.ipynb.
df_results.to_csv(OUT_DIR / "final_results.csv", index=False)
pd.DataFrame(fedavg_hist).to_csv(OUT_DIR / "fedavg_per_round.csv", index=False)
if best_strategy in fedavg_train_hist_per_mask:
    pd.DataFrame(fedavg_train_hist_per_mask[best_strategy]).to_csv(OUT_DIR / "fedavg_train_loss_per_round.csv", index=False)
pd.DataFrame(fedlr_hist).to_csv(OUT_DIR / "fedlr_per_round.csv", index=False)
if fedlr_train_hist:
    pd.DataFrame(fedlr_train_hist).to_csv(OUT_DIR / "fedlr_train_loss_per_round.csv", index=False)
pd.DataFrame(fedrf_hist).to_csv(OUT_DIR / "fedrf_per_round.csv", index=False)
for mu in FEDPROX_MUS:
    if mu in fedprox_histories:
        pd.DataFrame(fedprox_histories[mu]).to_csv(OUT_DIR / f"fedprox_mu{mu}_per_round.csv", index=False)
    if mu in fedprox_train_histories:
        pd.DataFrame(fedprox_train_histories[mu]).to_csv(OUT_DIR / f"fedprox_mu{mu}_train_loss_per_round.csv", index=False)

# Save best params
with open(OUT_DIR / "best_params_used.txt", "w") as f:
    f.write(f"DRUG={DRUG_NAME}\n")
    f.write(f"BEST_MASK={best_strategy}\n")
    f.write(f"BEST_MLP_LR={BEST_MLP_LR}\n")
    f.write(f"BEST_MLP_DH={BEST_MLP_DH}\n")
    f.write(f"BEST_MLP_THRESH={BEST_MLP_THRESH}\n")
    f.write(f"BEST_LR_C={BEST_LR_C}\n")
    f.write(f"BEST_LR_THRESH={BEST_LR_THRESH}\n")
    f.write(f"RF_PARAMS={RF_PARAMS}\n")
    f.write(f"BEST_RF_THRESH={BEST_RF_THRESH}\n")

# Archive
src_nb = Path("06-03c-Species-Masked-Ceftriaxone-Federated.ipynb")
if not src_nb.exists():
    import glob as _g
    candidates = list(_g.glob("/content/**/06-03c*.ipynb", recursive=True))
    if candidates: src_nb = Path(candidates[0])
if src_nb.exists():
    shutil.copy(str(src_nb), str(OUT_DIR / "notebook.ipynb"))
    print("Notebook archived.")

print(f"\n{'='*60}")
print(f"  Done. Results in {OUT_DIR.resolve()}")
print(f"  Best mask: {best_strategy}")
for f in sorted(OUT_DIR.glob("*")): print(f"    {f.name}")

---
**Done.** Species-masked federated analysis complete.

| Output | Location |
|---|---|
| RF species models | `rf_species_models/site_{A,B,C,D}_rf_species.joblib` |
| Masks | `masks/{union,majority,persite}/` |
| Results | `results/` |
| Models | `models/{strategy}/` |